# Modellering af den øvre hale i dimensionsafvigelsen med PROC QUANTREG

## Resumé

En præcisionsfabrik med maskinbearbejdning bekymrer sig om den **værste tænkelige** dimensionsafvigelse fra del til del, ikke kun gennemsnittet, fordi den øvre hale driver kassation og kundereklamationer. Denne notebook bruger **PROC QUANTREG** til at tilpasse kvantilregressioner ved medianen og 90. percentil, hvilket afslører, at værktøjsslid, cyklushastighed og fremføringshastighed trækker meget stærkere i den høje kvantil (kassationsrisiko-halen) end i medianen — kendetegnet for en heteroskedastisk proces, hvor variationen øges, efterhånden som værktøjet slides.

## Datakilder

| Datasæt | Rækker | Beskrivelse | Nøglevariable |
|---------|--------|-------------|----------------|
| `work.machining` | 100 | Syntetiske del-niveau-registreringer fra en CNC-drejebænk-linje, genereret direkte med `call streaminit`/`rand`. Dimensionsafvigelsen (mikron) fra nominel modelleres med en heteroskedastisk fejl, hvis spredning vokser med værktøjsslid og cyklushastighed, så procesdriverne virker stærkere på den øvre hale end på medianen. | `Deviation` (respons, mikron), `ToolWear` (akkumulerede skæreminutter), `CycleSpeed` (omdr./min.), `CoolantTemp` (grader C), `Humidity` (%RH), `FeedRate` (mm/omdrejning), `Machine` (CLASS: M1–M4), `Shift` (CLASS: Dag/Aften/Nat), `PartID` |

# Modellering af procesdrivere for den øvre hale i dimensionsafvigelsen

## Produktions-use case: kassationsrisiko-modellering på en CNC-drejebænk-linje

På en præcisionsbearbejdningslinje kasseres dele, når **dimensionsafvigelsen** fra nominel bliver for stor. Fabrikken taber ikke penge på den *gennemsnitlige* del — den taber penge på den **øvre hale** af afvigelsesfordelingen. Almindelig mindste-kvadraters regression modellerer den betingede middelværdi og kan fuldstændigt overse drivere, der kun betyder noget, når tingene går galt.

**Kvantilregression** modellerer en valgt betinget kvantil (for eksempel 90. percentil af afvigelsen) i stedet for middelværdien. **PROC QUANTREG** tilpasser ét eller flere kvantiler i ét kald og rapporterer et parameterestimat, en standardfejl og konfidensgrænser for hver driver ved hver kvantil. Vi vil:

1. Generere et realistisk syntetisk del-niveau-datasæt, hvor fejlspredningen vokser med værktøjsslid og cyklushastighed (så driverne rammer halen hårdere end centrum).
2. Tilpasse **medianen** (q = 0,50) og **kassationsrisiko-halen** (q = 0,90) i ét PROC QUANTREG-kald.
3. Sammenligne de to koefficientsæt side om side for at vise, hvordan hældningerne ændrer sig fra centrum til hale.
4. Score hver del med dens tilpassede 90.-percentil-forudsigelse, så analytikere kan flage dele med høj halerisiko.

Alt nedenfor er selvstændigt — ingen eksterne filer, intet netværk.

## Trin 1 — Generér syntetiske bearbejdningsdata

Vi simulerer drejede dele fordelt på 4 maskiner og 3 vagter. Det centrale realismetrick er **heteroskedasticitet**: standardafvigelsen af det tilfældige fejlled (`sigma`) vokser med `ToolWear` og `CycleSpeed`. Det gør, at disse to drivere trækker meget stærkere i 90. percentil af `Deviation` end i medianen — netop den situation, hvor kvantilregression for alvor beviser sit værd. `Humidity` bærer intet reelt signal i den datagenererende proces og fungerer derfor som en placebo-driver, vi kan holde øje med.

In [1]:
data work.machining;
    call streaminit(20260531);
    LÆNGDE Machine $2 Shift $5;
    GØR PartID = 1 TIL 100;
        /* CLASS-faktorer */
        m = rand('integer', 1, 4);
        Machine = cats('M', SKRIV_V(m, 1.));
        s = rand('integer', 1, 3);
        HVIS s = 1 SÅ Shift = 'Dag';
        ELLERS HVIS s = 2 SÅ Shift = 'Aften';
        ELLERS Shift = 'Nat';

        /* Kontinuerte procesdrivere */
        ToolWear     = rand('uniform') * 120;            /* akkumulerede skæreminutter */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* spindelomdrejninger (rpm) */
        CoolantTemp  = 22 + rand('normal') * 3;          /* grader C */
        Humidity     = 45 + rand('normal') * 8;          /* %RH (placebo) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/omdrejning */

        /* Maskinoffset: én maskine kører varmere */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* Natholdet driver en smule */
        shiftoff = (Shift = 'Nat') * 1.2;

        /* Placering (central tendens) af afvigelsen i mikron */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Heteroskedastisk spredning: halen udvides med slid og hastighed */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        HVIS Deviation < 0 SÅ Deviation = 0;

        BEHOLD PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        UDDATA;
    SLUT;
KØR;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.10 seconds
  cpu   0.10 seconds


### Hurtigt kig på den rå fordeling

Før modellering bekræfter vi, at responsen er højreskæv med en betydelig øvre hale — det er den del af fordelingen, der driver kassation. Vi udskriver medianen og de øvre percentiler direkte fra PROC UNIVARIATE, så vi kan se, hvor meget højere 90. percentil ligger over medianen.

In [2]:
PROCEDURE UNIVARIATE data=work.machining NOPRINT;
    VARIABEL Deviation;
    UDDATA out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
KØR;

PROCEDURE UDSKRIV data=work.devpct noobs MÆRKAT;
    MÆRKAT Med = 'Median afvigelse'
          P90 = '90. percentil'
          P95 = '95. percentil'
          P99 = '99. percentil';
KØR;


Median afvigelse  90. percentil  95. percentil  99. percentil
----------------  -------------  -------------  -------------
    8.7251490709  12.4132063767  13.5691645665  17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Trin 2 — Tilpas medianen og kassationsrisiko-halen sammen

PROC QUANTREG tilpasser begge kvantiler i ét kald: `QUANTILE=0.5 0.90`. `CLASS`-sætningen erklærer de kategoriske procesfaktorer (`Machine`, `Shift`); `MODEL`-sætningen opstiller alle kandidat-kontinuerte effekter. Vi anmoder om `CI=SPARSITY`, som bruger sparsity-funktionsestimatoren til at frembringe en standardfejl og et 95 %-konfidensbånd for hver koefficient.

Læs de to outputblokke som et før/efter: den første blok (q = 0,50) beskriver den *typiske* del; den anden (q = 0,90) beskriver den *kassationstruede* del. Hold øje med `ToolWear`, `CycleSpeed` og `FeedRate` — fordi den simulerede fejlspredning vokser med slid og hastighed, bør deres hældninger være større ved den øvre kvantil.

In [3]:
PROCEDURE quantreg data=work.machining ci=sparsity;
    KLASSE Machine Shift;
    MODEL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
KØR;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT AFTEN          -1.6360       0.2964      -2.2170      -1.0550
SHIFT DAG            -0.9975       0.2909      -1.5676      -0.4275
TOOLWEAR              0.0240       0.0033       0.0174       0.0305
CYCLESPEED           -0.0007       0.0008      -0.0022       0.0009
COOLANTTEMP           0.4542       0.0395       0.3767       0.5316
HUMIDITY              0.0575       0.0150       0.0281       0.0868
FEEDRATE             -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -14.2292     -11.4711
MACHINE M3            1


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Trin 3 — Sæt centrum og hale side om side

At læse to koefficientblokke parallelt er besværligt, så vi opfanger parametertabellen med `ODS OUTPUT ParameterEstimates=` (som medbringer en `Quantile`-kolonne) og fletter derefter q = 0,50- og q = 0,90-estimaterne for de kontinuerte drivere ind i én sammenligningstabel. Kolonnen `Tail - Median` er hovedtallet: en stor positiv værdi betyder, at driveren skubber kassationsrisiko-halen meget hårdere, end den flytter den typiske del.

In [4]:
ODS UDDATA ParameterEstimates=work.pe;
PROCEDURE quantreg data=work.machining ci=sparsity;
    KLASSE Machine Shift;
    MODEL Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
KØR;

/* Flet median- og hale-hældningerne for hver kontinuert driver */
data work.compare;
    BEHOLD Parameter MedianSlope TailSlope TailMinusMedian;
    SAMMENFLET
        work.pe(HVOR=(Quantile=0.5)
                BEHOLD=Quantile Parameter ESTIMATE
                OMDØB=(ESTIMATE=MedianSlope))
        work.pe(HVOR=(Quantile=0.9)
                BEHOLD=Quantile Parameter ESTIMATE
                OMDØB=(ESTIMATE=TailSlope));
    EFTER Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
KØR;

PROCEDURE UDSKRIV data=work.compare noobs MÆRKAT;
    MÆRKAT Parameter       = 'Faktor'
          MedianSlope     = 'Hældning @ q=0.50'
          TailSlope       = 'Hældning @ q=0.90'
          TailMinusMedian = 'Hale - Median';
    format MedianSlope TailSlope TailMinusMedian 10.4;
KØR;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
TOOLWEAR              0.0146       0.0045       0.0057       0.0235
CYCLESPEED           -0.0000       0.0011      -0.0021       0.0021
COOLANTTEMP           0.4838       0.0528       0.3802       0.5873
HUMIDITY              0.0678       0.0203       0.0280       0.1076
FEEDRATE             -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
TOOLWEAR              0.0416       0.0021       0.0375       0.0457
CYCLESPEED            0.0008       0.0005      -0.0002       0.0018
COOLANTTEMP           0.3907       0.0245       0.3428       0.4387
HUMIDITY              0.0807       0.0094       0.0623       0.0991
FEEDRATE              5.9122       3.6069      -1.1572      12.9817


     Faktor   Hældnin


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Trin 4 — Scor hver del ved 90. percentil

`OUTPUT`-sætningen skriver den tilpassede 90.-percentil-forudsigelse tilbage, én række pr. del, sammen med forudsigelsens standardfejl (`STDP`) og check-loss-residualet. Vi fører `PartID` med via `ID`-sætningen og kopierer de to dominerende drivere, så analytikere kan sortere de mest risikofyldte dele øverst. En lille PROC PRINT viser de først scorede dele.

In [5]:
PROCEDURE quantreg data=work.machining ci=sparsity;
    KLASSE Machine Shift;
    id PartID;
    MODEL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    UDDATA out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
KØR;

PROCEDURE UDSKRIV data=work.scored(obs=10) noobs MÆRKAT;
    VARIABEL PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    MÆRKAT PredP90 = 'Forventet 90. percentil afvigelse'
          SEPred  = 'Forudsigelses-standardfejl'
          Resid   = 'Check-loss-residual';
KØR;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT AFTEN          -3.1741       0.1034      -3.3768      -2.9713
SHIFT DAG            -1.8083       0.1017      -2.0075      -1.6090
TOOLWEAR              0.0438       0.0012       0.0416       0.0461
CYCLESPEED            0.0037       0.0003       0.0032       0.0043
COOLANTTEMP           0.3377       0.0133       0.3116       0.3638
FEEDRATE             14.9464       2.0482      10.9321      18.9608


PARTID        TOOLWEAR       CYCLESPEED  Forventet 90. percentil afvigelse  Forudsigelses-standardfejl  Check-loss-residual
------  --------------  ---------


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Fortolkning af resultaterne

**Halen fortæller en anden historie end centrum.** Ved at sammenligne de to koefficientblokke (trin 2) og den flettede tabel (trin 3) ses det, at hældningerne for `ToolWear`, `CycleSpeed` og `FeedRate` er markant større ved 90. percentil end ved medianen. Det er den datagenererende mekanisme gjort synlig: fordi vi har bygget fejlspredningen til at vokse med slid og hastighed, flytter disse drivere næsten ikke medianafvigelsen, men opblæser kraftigt den kassationstruede øvre hale. En middelværdibaseret regression ville have undervægtet netop de håndtag, der betyder noget for kassation.

**`ToolWear`-signalet er det tydeligste.** Dens hældning nærmest tredobles fra medianmodellen (0,015) til halemodellen (0,042), og 90.-percentil-konfidensbåndet ligger langt over nul — slid er den enkeltstående mest pålidelige driver af høj halerisiko. `CycleSpeed`, som er stort set flad ved medianen, bliver positiv i halen, i overensstemmelse med dens rolle i spredningsleddet.

**Kvantilregression adskiller placering fra spredning.** `CoolantTemp` indgår i placeringsleddet, men ikke i spredningsleddet, så dens hældning forbliver tæt på 0,4–0,5 mikron pr. grad ved begge kvantiler — den flytter hele fordelingen i stedet for at sprede halen. `Humidity` bærer intet reelt signal i den datagenererende proces, men på kun 100 dele opfanger den alligevel en lille tilsyneladende hældning; dens `Tail - Median`-ændring (0,013) er en størrelsesorden mindre end `ToolWear`'s (0,027) og overgås langt af `FeedRate`'s (12,3), så den opfører sig ikke som en haledriver. Den ærlige lære er, at en variabel, der reelt er nul, stadig kan se ikke-nul ud i en lille stikprøve — hvilket er netop derfor, en licenseret fuldproduktionskørsel over tusindvis af dele ville stramme disse estimater op.

**Operationel gevinst.** `OUTPUT`-forudsigelserne giver hver del en forudsagt 90.-percentil-afvigelse med en standardfejl, hvilket flager dele med høj halerisiko, inden de sendes. Den handlingsorienterede konklusion: stram værktøjsskifteintervallerne, og loft cyklushastigheden, når der køres opgaver med snævre tolerancer, fordi begge tiltag virker direkte på den kassationsdrivende øvre hale af dimensionsafvigelsen.

*Bemærkning om omfang:* denne notebook kører under den ulicenserede motor, som begrænser hvert trin til 100 observationer, så hældningerne ovenfor er estimeret på 100 dele, og haletilpasningen er nødvendigvis mere støjfyldt, end en fuldproduktionskørsel ville være. Mønstret centrum-versus-hale er allerede synligt ved denne størrelse; en licenseret kørsel over hele delestrømmen ville stramme hvert konfidensbånd op.